# Поиск мест через Geoapify
Из названия и контекста получаем кандидатов с адресами и координатами. Сравниваем языки, филиалы, JSON/GeoJSON и MCP. Это ручные поисковые формулировки для японских мест из наших примеров, включая Tower Records с кадра De2M3kBN9Uf. Полная фраза ниже искусственная. Автоматическое извлечение LLM здесь ещё предстоит подключить.


In [1]:
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/geoapify_places.py").exists())
sys.path.insert(0, str(ROOT))
from src.geoapify_places import GeoapifyClient, load_api_key, candidates, candidate_rows

client = GeoapifyClient(load_api_key(ROOT / ".env"))
records = []
print("Ключ загружен. Python", platform.python_version())
print("Повторный Run All снова расходует квоту API.")

def show(record):
    rows = candidate_rows(record)
    print("Кандидатов:", len(rows), "| HTTP, с:", record["elapsed_seconds"])
    for row in rows[:3]:
        print(json.dumps({k: row[k] for k in ("rank", "name", "address", "confidence", "lat", "lon", "distance_m")}, ensure_ascii=False))

Ключ загружен. Python 3.12.3
Повторный Run All снова расходует квоту API.


## Язык, контекст и неоднозначность филиалов
Для всех запросов явно задана Япония. Язык ответа en управляет отображением результата. Сравниваем русскую фразу, кириллическое название, латиницу и японскую вывеску. Сохраняем все первые пять кандидатов.

In [2]:
queries = [
    "В Сибуе зашли в Тауэр Рекордс за пластинками",
    "Тауэр Рекордс, Сибуя, Токио",
    "Tower Records",
    "Tower Records, Shibuya, Tokyo",
    "タワーレコード渋谷店",
    "Tower Records Shinjuku",
    "Ikema Bridge, Okinawa",
    "Sekizenkan, Gunma",
]
for query in queries:
    record = {"query": query, **client.geocode(query)}
    records.append(record)
    print("\nЗапрос:", query)
    show(record)


Запрос: В Сибуе зашли в Тауэр Рекордс за пластинками
Кандидатов: 0 | HTTP, с: 0.483

Запрос: Тауэр Рекордс, Сибуя, Токио
Кандидатов: 0 | HTTP, с: 0.523

Запрос: Tower Records
Кандидатов: 5 | HTTP, с: 1.548
{"rank": 1, "name": "Tower Records", "address": "Tower Records, 14, Jinnan, Shibuya, Jinnan 1 150-0041, Japan", "confidence": 1, "lat": 35.6618899, "lon": 139.7010954, "distance_m": null}
{"rank": 2, "name": "Tower Records", "address": "Tower Records, 1, Shinjuku, Shinjuku, Shinjuku 3 160-0022, Japan", "confidence": 1, "lat": 35.6900537, "lon": 139.7015777, "distance_m": null}
{"rank": 3, "name": "Tower Records", "address": "Tower Records, Okachocho Station Boulevard, Ueno, Taito, Ueno 6 110-0005, Japan", "confidence": 1, "lat": 35.7107205, "lon": 139.7749294, "distance_m": null}

Запрос: Tower Records, Shibuya, Tokyo
Кандидатов: 1 | HTTP, с: 0.849
{"rank": 1, "name": "Tower Records", "address": "Tower Records, 14, Jinnan, Shibuya, Jinnan 1 150-0041, Japan", "confidence": 0.8, "lat"

## JSON, GeoJSON и MCP
Повторяем точный запрос с районом через два дополнительных интерфейса. Сравниваем координаты и place_id. Такое совпадение проверяет интеграцию интерфейсов одной базы.

In [3]:
query = "Tower Records, Shibuya, Tokyo"
rest = next(r for r in records if r["query"] == query)
geojson = {"query": query, **client.geocode(query, response_format="geojson")}
mcp = {"query": query, **client.mcp_geocode(query)}
records.extend([geojson, mcp])
for label, record in [("REST JSON", rest), ("REST GeoJSON", geojson), ("MCP", mcp)]:
    print("\n", label)
    show(record)
    print("place_id:", candidate_rows(record)[0]["place_id"])

identities = [(candidates(r)[0]["lat"], candidates(r)[0]["lon"], candidates(r)[0]["place_id"]) for r in [rest, geojson, mcp]]
print("Координаты и ID совпали:", len(set(identities)) == 1)
assert len(set(identities)) == 1


 REST JSON
Кандидатов: 1 | HTTP, с: 0.849
{"rank": 1, "name": "Tower Records", "address": "Tower Records, 14, Jinnan, Shibuya, Jinnan 1 150-0041, Japan", "confidence": 0.8, "lat": 35.6618899, "lon": 139.7010954, "distance_m": null}
place_id: 5109cc9e5f6f76614059bf06e9ceb8d44140f00102f901be55260800000000c0020192030d546f776572205265636f726473

 REST GeoJSON
Кандидатов: 1 | HTTP, с: 0.648
{"rank": 1, "name": "Tower Records", "address": "Tower Records, 14, Jinnan, Shibuya, Jinnan 1 150-0041, Japan", "confidence": 0.8, "lat": 35.6618899, "lon": 139.7010954, "distance_m": null}
place_id: 5109cc9e5f6f76614059bf06e9ceb8d44140f00102f901be55260800000000c0020192030d546f776572205265636f726473

 MCP
Кандидатов: 1 | HTTP, с: 0.627
{"rank": 1, "name": "Tower Records", "address": "Tower Records, 14, Jinnan, Shibuya, Jinnan 1 150-0041, Japan", "confidence": 0.8, "lat": 35.6618899, "lon": 139.7010954, "distance_m": null}
place_id: 5109cc9e5f6f76614059bf06e9ceb8d44140f00102f901be55260800000000c002019203

## Кафе рядом и сохранение результата
Берём первый результат запроса с явно указанным районом только для демонстрации. В сервисе этому шагу должно предшествовать подтверждение места. Радиус поиска 200 м, максимум четыре кафе. Сохраняем ответы и метрики в outputs, ключ в результаты не входит.

In [4]:
place = candidates(rest)[0]
nearby = {"query": "catering.cafe, radius 200 m", **client.nearby(place["lat"], place["lon"])}
records.append(nearby)
show(nearby)

summary = [
    {"query": r["query"], "endpoint": r["endpoint"], "candidates": len(candidates(r)),
     "seconds": r["elapsed_seconds"], "attempts": r["attempts"]}
    for r in records
]
for row in summary:
    print(json.dumps(row, ensure_ascii=False))
print("Записей запросов:", len(records))
print("Всего попыток HTTP:", sum(r["attempts"] for r in records))
print("Сумма HTTP, с:", round(sum(r["elapsed_seconds"] for r in records), 3))
stamp = datetime.now(timezone.utc)
destination = ROOT / "outputs" / "geoapify_places" / (stamp.strftime("%Y%m%dT%H%M%S%fZ") + ".json")
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps({
    "retrieved_at_utc": stamp.isoformat(), "country_filter": "jp", "response_language": "en",
    "python_version": platform.python_version(), "records": records,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("Ответы сохранены:", destination.relative_to(ROOT).as_posix())

Кандидатов: 4 | HTTP, с: 0.508
{"rank": 1, "name": "atari CAFE&DINING", "address": "atari CAFE&DINING, 3, Jinnan, Shibuya, Jinnan 1 150-0041, Japan", "confidence": null, "lat": 35.6617137, "lon": 139.7005995, "distance_m": 49}
{"rank": 2, "name": "スターバックス", "address": "Starbucks, 3, Jinnan, Shibuya, Jinnan 1 150-0041, Japan", "confidence": null, "lat": 35.6614154, "lon": 139.7006238, "distance_m": 68}
{"rank": 3, "name": "ChugaiGraceCAFE", "address": "ChugaiGraceCAFE, 3, Jinnan, Shibuya, Jinnan 1 150-0041, Japan", "confidence": null, "lat": 35.6614338, "lon": 139.7005688, "distance_m": 70}
{"query": "В Сибуе зашли в Тауэр Рекордс за пластинками", "endpoint": "v1/geocode/search", "candidates": 0, "seconds": 0.483, "attempts": 1}
{"query": "Тауэр Рекордс, Сибуя, Токио", "endpoint": "v1/geocode/search", "candidates": 0, "seconds": 0.523, "attempts": 1}
{"query": "Tower Records", "endpoint": "v1/geocode/search", "candidates": 5, "seconds": 1.548, "attempts": 1}
{"query": "Tower Records, Sh

## Интерпретация и следующий шаг
В сохранённом запуске русские запросы не находят магазин, латиница с районом и японское название дают одну точку. Общий запрос возвращает несколько филиалов с confidence 1. У моста и гостиницы confidence ниже, поэтому единый высокий порог отсечёт полезных кандидатов. Эти наблюдения относятся к ручным примерам. Адрес магазина в базе может быть неполным.

Следующий эксперимент — извлекать упоминания с таймкодами и сопоставлять кандидатов со свидетельствами сцены. Шаблоны промптов приведены в [гайде](geoapify_places.md). Географическая точность по размеченному корпусу здесь ещё не измерена.

Данные предоставлены Geoapify и OpenStreetMap contributors. [Документация Geoapify](https://apidocs.geoapify.com/docs/geocoding/forward-geocoding/), [MCP](https://apidocs.geoapify.com/docs/mcp).